# Local Models with LM Studio

Run an open-weight model on your own machine with LM Studio and call it with the OpenAI SDK by changing only the base URL.

## Before you start

This notebook calls a server on your own computer, so run it locally (Jupyter or VS Code), not in Colab.

1. Download LM Studio from https://lmstudio.ai and open it.
2. Download a model from the **Discover** tab. A small instruction-tuned model (1B to 8B parameters) is a good start on a laptop.
3. Start the local server from the **Developer** tab, or from a terminal:

```bash
lms server start
lms load <model-name>
```

The server listens on `http://localhost:1234` and speaks the same API as OpenAI.

In [1]:
%pip install openai langchain-openai --upgrade --quiet

## Point the OpenAI SDK at LM Studio

Only two things change compared with calling OpenAI:

- `base_url` points at your local server.
- `api_key` can be any placeholder: the local server does not check it.

In [2]:
from openai import OpenAI

local = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

models = [m.id for m in local.models.list().data]
print(models)

['google/gemma-3-1b', 'qwen/qwen3-4b-2507', 'text-embedding-nomic-embed-text-v1.5']


In [3]:
# Set this to the identifier LM Studio shows for the model you downloaded:
MODEL = "google/gemma-3-1b"
assert MODEL in models, f"{MODEL} is not available in LM Studio. Pick one of: {models}"

In [4]:
response = local.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise marketing assistant."},
        {"role": "user", "content": "Give me three blog post ideas about digital marketing."},
    ],
)
print(response.choices[0].message.content)

Okay, here are three concise blog post ideas about digital marketing:

1. **Headline:** "Boost Your Website Traffic with SEO – A Beginner's Guide"
   * **Content Focus:** Explain the basics of SEO (Search Engine Optimization) – keyword research, on-page optimization, and link building.  Keep it simple and actionable.
   * **Keywords:** SEO, Keyword Research, Website Optimization

2. **Headline:** "Instagram Reels – Your Secret Weapon for Engagement"
   * **Content Focus:** Demonstrate how to create engaging short-form video content for Instagram Reels. Tips on using trending sounds, captions, and calls to action.
   * **Keywords:** Instagram Reels, Short-Form Video, Engagement

3. **Headline:** "Email Marketing That Actually Works (Without the Spam!)"
   * **Content Focus:** Offer practical advice for building an effective email marketing strategy – segmentation, personalization, and A/B testing.
   * **Keywords:** Email Marketing, Segmentation, Personalization

---

Would you like me 

## Structured outputs from a local model

LM Studio supports JSON schema structured outputs, so the same Pydantic pattern works locally. We generate an article outline.

In [5]:
from pydantic import BaseModel


class Section(BaseModel):
    title: str


class ArticleOutline(BaseModel):
    title: str
    sections: list[Section]


completion = local.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an SEO specialist who writes article outlines."},
        {"role": "user", "content": "Create an article outline about digital marketing for small businesses."},
    ],
    response_format=ArticleOutline,
)

outline = completion.choices[0].message.parsed
print(outline.title)
for section in outline.sections:
    print("-", section.title)

Digital Marketing for Small Businesses: A Beginner's Guide - Drive Growth Without Breaking the Bank
- Introduction - The Digital Landscape & Why It Matters for Small Businesses
- Understanding Your Target Audience - Who Are You Trying to Reach?
- Budgeting & Resources – Where Do You Start?
- Essential Digital Channels - A Breakdown
- Content is King – Creating Valuable Content for Your Customers
- Social Media Strategy - More Than Just Posting
- Email Marketing – Nurturing Leads & Sales
- Local SEO - Getting Found in Your Community
- Measuring & Analyzing Results – Show That You're Working!
- Advanced Strategies - Taking Your Digital Marketing to the Next Level
- Free Tools & Resources


## The same idea in LangChain

Any library built on the OpenAI API can use a local model the same way. In LangChain, `ChatOpenAI` takes the same `base_url`.

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", model=MODEL, temperature=0)

structured_llm = llm.with_structured_output(ArticleOutline, method="json_schema")
langchain_outline = structured_llm.invoke("Create an article outline about email marketing.")
langchain_outline

ArticleOutline(title='Mastering Email Marketing: A Comprehensive Guide for [Your Target Audience/Industry]', sections=[Section(title='Introduction - Why Email Marketing Matters (200-300 words)**: **')])

## Swapping between local and hosted

Because the API is the same, moving between a local model and a hosted one is a configuration change. Keep the base URL, key and model name in one place:

In [7]:
import os

PROVIDERS = {
    "local": {"base_url": "http://localhost:1234/v1", "api_key": "lm-studio", "model": MODEL},
    # "openai": {"base_url": "https://api.openai.com/v1", "api_key": os.environ.get("OPENAI_API_KEY"), "model": "gpt-5.6-luna"},
    # "openrouter": {"base_url": "https://openrouter.ai/api/v1", "api_key": os.environ.get("OPENROUTER_API_KEY"), "model": "openai/gpt-5.6-luna"},
}

provider = PROVIDERS["local"]
client = OpenAI(base_url=provider["base_url"], api_key=provider["api_key"])

reply = client.chat.completions.create(
    model=provider["model"],
    messages=[{"role": "user", "content": "In one sentence, why might a company run an AI model locally?"}],
)
print(reply.choices[0].message.content)

Companies often run AI models locally to improve privacy, reduce reliance on cloud services, and potentially optimize performance for specific use cases due to data sensitivity or resource constraints.


## Summary

- LM Studio runs open-weight models on your own machine and exposes an OpenAI-compatible server on `localhost:1234`.
- Calling it with the OpenAI SDK only needs a different `base_url` and a placeholder `api_key`.
- Structured outputs work locally too, so the Pydantic patterns from the hosted API carry over.
- Local models keep data on your device and cost nothing per token, but smaller models are less capable and depend on your hardware.
- Keeping the base URL, key and model in configuration lets you switch between local and hosted models without code changes.